> **Release dependency:** the original `clinical_review/` helper package was absent from the supplied source tree. This notebook is retained for documentation but cannot run until those original helpers are restored. See `docs/REPRODUCIBILITY.md`. No completed reviewer data are distributed.

# BR08C — Export all required adjudications for offline review

Run after initial ratings are frozen and BR08 has created `adjudication_queue.json`.
This creates one JSON containing every required report adjudication, not just the
first pending item. No clinical decisions are supplied. Existing completed entries
are preserved. Previously submitted native adjudications are copied unchanged if a
new form is needed. No model, GPU or PULSE-specific kernel is required.

Default package: Shaoxing PULSE/base-Llama calibration. Use the exact package you prepared.


In [ ]:
from pathlib import Path
import sys
EXP = Path('/path/to/repository/experiment02')
REVIEW_DIR = Path(
    '/path/to/repository/outputs/real_run_001/'
    'experiment02_v1/context_fit/v1/shaoxing/clinical_review/shaoxing/'
    'calibration_pulse_llama_base_v1'
)
sys.path.insert(0, str(EXP))
from common import read, sha, signature, immutable, lock, atomic
from clinical_review.workflow import _check, RATING_FIELDS, progress
print('Review package:', REVIEW_DIR)
print('Progress:', progress(REVIEW_DIR))


In [ ]:
def by_id(rows, key):
    if not isinstance(rows, list) or not all(isinstance(r, dict) for r in rows):
        raise ValueError('Expected a list of records: ' + key)
    ids = [r.get(key) for r in rows]
    if any(not isinstance(x, str) or not x for x in ids) or len(set(ids)) != len(ids):
        raise ValueError('Missing/duplicate IDs: ' + key)
    return dict(zip(ids, rows))


def export_adjudication_form():
    public = REVIEW_DIR / 'PUBLIC'
    target = public / 'adjudications_completed.json'
    with lock(REVIEW_DIR / 'offline_adjudication_form.lock'):
        protocol = _check(REVIEW_DIR)
        ap = read(REVIEW_DIR / 'PRIVATE/adjudication_protocol.json')
        for field, name in (
            ('initial_sha256', 'PRIVATE/initial_submitted.json'),
            ('references_sha256', 'PRIVATE/references_submitted.json'),
            ('queue_sha256', 'PUBLIC/adjudication_queue.json'),
        ):
            if ap.get(field) != sha(REVIEW_DIR / name):
                raise ValueError('Frozen adjudication source changed: ' + name)
        if ap.get('reviewer_id') != protocol['reviewer_id']:
            raise ValueError('Adjudication reviewer differs from package')
        queue = read(public / 'adjudication_queue.json')
        queued = by_id(queue, 'review_id')
        initial = by_id(read(REVIEW_DIR / 'PRIVATE/initial_submitted.json'), 'review_id')
        refs = by_id(read(REVIEW_DIR / 'PRIVATE/references_submitted.json'), 'case_id')
        reports = by_id(read(public / 'reports.json'), 'review_id')
        if set(initial) != set(reports):
            raise ValueError('Initial ratings do not cover all released reports')
        if not queue:
            return dict(required=0, instruction='No adjudications required; check progress and continue to BR09.')
        rows = []
        images = {}
        saved_count = 0
        for rid, item in queued.items():
            if item['initial_rating'] != initial.get(rid):
                raise ValueError('Queue differs from frozen rating: ' + rid)
            if item['independent_interpretation'] != refs.get(item['case_id']):
                raise ValueError('Queue differs from frozen interpretation: ' + rid)
            if rid not in reports or any(item.get(k) != reports[rid].get(k)
                    for k in ('case_id', 'image', 'generated_report')):
                raise ValueError('Queue differs from released report: ' + rid)
            image = (public / item['image']).resolve()
            if public.resolve() not in image.parents:
                raise ValueError('Image is outside PUBLIC')
            images[item['image']] = sha(image)
            row = dict(review_id=rid, case_id=item['case_id'], image=item['image'],
                reviewer_id=protocol['reviewer_id'], initial_rating_sha256=signature(item['initial_rating']),
                decision=None, reason='', evidence_reviewed='', final_rating=None)
            saved_path = public / 'adjudications' / (rid + '.json')
            if saved_path.exists():
                saved = read(saved_path)
                if any(saved.get(k) != row[k] for k in ('review_id', 'reviewer_id', 'initial_rating_sha256')):
                    raise ValueError('Saved adjudication has different identity: ' + rid)
                for k in ('decision', 'reason', 'evidence_reviewed', 'final_rating'):
                    row[k] = saved[k]
                saved_count += 1
            rows.append(row)
        identity = dict(version='offline_adjudication_source_v1',
            package_protocol_sha256=sha(REVIEW_DIR / 'PRIVATE/protocol.json'),
            adjudication_protocol_sha256=sha(REVIEW_DIR / 'PRIVATE/adjudication_protocol.json'),
            queue_signature=signature(queue), image_hashes=images)
        form = dict(schema_version='blinded_report_adjudications_v1', stage=protocol['stage'],
            source_queue_file='adjudication_queue.json', source_queue_signature=signature(queue),
            expected_adjudications=len(rows), adjudications=rows)
        # Never replace a partially or fully completed form.
        if target.exists():
            existing = read(target)
            if any(existing.get(k) != form[k] for k in form if k != 'adjudications'):
                raise ValueError('Existing form belongs to a different queue; preserve it')
            uploaded = by_id(existing.get('adjudications'), 'review_id')
            if set(uploaded) != set(queued):
                raise ValueError('Existing form has different IDs; preserve it')
            for row in rows:
                old = uploaded[row['review_id']]
                if set(old) != set(row) or any(old[k] != row[k] for k in (
                        'review_id', 'case_id', 'image', 'reviewer_id', 'initial_rating_sha256')):
                    raise ValueError('Existing form identity changed; preserve it')
        immutable(REVIEW_DIR / 'PRIVATE/offline_adjudication_source.json', identity)
        existed = target.exists()
        if not existed:
            atomic(target, form)
        return dict(required=len(rows), previously_saved=saved_count,
            existing_form_preserved=existed, form=str(target),
            queue=str(public / 'adjudication_queue.json'), images=str(public / 'images'))

print(export_adjudication_form())


In [ ]:
PUBLIC = REVIEW_DIR / 'PUBLIC'
GUIDE = PUBLIC / 'ADJUDICATION_INSTRUCTIONS.txt'
text = """SECOND-PASS ADJUDICATION — SAME REVIEWER

Review every entry in adjudications_completed.json alongside the matching
review_id in adjudication_queue.json and its ECG image. The queue contains
your frozen independent interpretation, generated report and initial rating.
Review IDs identify reports, while several reports can share one case/image.
Only flagged reports are queued; the queue can contain fewer than 40 entries.

Fill decision, reason and evidence_reviewed for every pending entry:
- retain_initial: confirm the initial rating; leave final_rating null.
- revise_initial: supply a complete final_rating object with ALL rating fields.
  Copy those fields from the initial rating and make your own justified changes.
- unresolved: explain remaining uncertainty; leave final_rating null.

Do not alter IDs, image paths, source signatures, reviewer ID or queue metadata.
Do not edit frozen interpretations or initial ratings. A confirmed disagreement
is a valid result; do not revise merely to agree with a generated report.
Describe the ECG features and other permitted evidence actually rechecked.
A later review session is preferable. This is same-reviewer adjudication,
not independent validation or inter-rater reliability.

Upload the completed adjudications_completed.json to the SAME PUBLIC folder.
Run BR08D_import_exported_adjudications.ipynb, then BR09 for this review version.
Do not use the older BR08BS with constants tied to the earlier model package.
Do not edit files while an import is running. Keep PRIVATE on Biowulf.
"""
text += '\nRequired final_rating fields for revise_initial: ' + ', '.join(RATING_FIELDS) + '\n'
if GUIDE.exists() and GUIDE.read_text(encoding='utf-8') != text:
    raise ValueError('Existing instructions differ; preserve and inspect them')
if not GUIDE.exists():
    GUIDE.write_text(text, encoding='utf-8')
print('DOWNLOAD:', PUBLIC / 'adjudications_completed.json')
print('ALSO DOWNLOAD: adjudication_queue.json, images/, ADJUDICATION_INSTRUCTIONS.txt')
print('UPLOAD completed file to:', PUBLIC / 'adjudications_completed.json')
print('NEXT: BR08D with this same REVIEW_DIR, after clinician completion.')
